# Compare and Evaluate Chunking Strategies

In this notebook, we walk through a standard practice for evaluating competing chunking strategies using AI Observability.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
import os

os.environ["TRULENS_OTEL_TRACING"] = "1"

## Step 1: Setup resources and load data.

Start by running the [setup.sql](./setup.sql) in your account with your username filled in at the top as `<YOUR_USER>`.

Then, download the pdf documents from the [data folder in this GitHub repository](https://github.com/Snowflake-Labs/sfguide-compare-and-evaluate-chunking-strategies/tree/main/data) and upload them to the stage `PDF_10KS` you just created.

## Step 2: Parse Documents

Next, you will parse the content of uploaded PDFs using Snowflake's [AI_PARSE_DOCUMENT](https://docs.snowflake.com/sql-reference/functions/parse_document-snowflake-cortex). We will use Layout mode to capture rich elements of the documents such as embedded tables, columns and pull quotes.

Objective: **Transform raw PDFs into parsed text, so it can be split and prepared for efficient search and retrieval.**

Key Outputs:
- CHUNKING_EVALUATION.DATA.PARSED_TEXT: Table containing the raw text.

Note: the documents are relatively large, so this step can take some time.



In [ ]:
DB_NAME = 'CHUNKING_EVALUATION'
DOC_SCHEMA_NAME = 'DOCS'
TEXT_SCHEMA_NAME = 'PARSED_DATA'

In [ ]:
-- Create a table to hold the extracted text from the PDF files
CREATE TABLE {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARSED_TEXT (relative_path VARCHAR(500), raw_text VARIANT) IF NOT EXISTS;

In [ ]:
INSERT INTO {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARSED_TEXT (relative_path, raw_text)
WITH pdf_files AS (
    SELECT DISTINCT
        METADATA$FILENAME AS relative_path
    FROM @{{DB_NAME}}.{{DOC_SCHEMA_NAME}}.PDF_10KS
    WHERE METADATA$FILENAME ILIKE '%.pdf'
      -- Exclude files that have already been parsed
      AND METADATA$FILENAME NOT IN (SELECT relative_path FROM {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARSED_TEXT)
)
SELECT 
    relative_path,
    SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
        '@{{DB_NAME}}.{{DOC_SCHEMA_NAME}}.PDF_10KS',  -- Your stage name
        relative_path,  -- File path
        {'mode': 'layout'}  -- Adjust mode as needed ('layout', 'ocr')
    ) AS raw_text
FROM pdf_files;

## Step 1: Split (Chunk) text

Next, we will chunk the parsed text using [SPLIT_TEXT_RECURSIVE_CHARACTER](https://docs.snowflake.com/sql-reference/functions/split_text_recursive_character-snowflake-cortex).

Objectives: 
- Split the parsed documents into manageable chunks, prepared for efficient search and retrieval.
- Add key metadata to the chunks
- Contextualize the chunks with document summaries

Key Outputs:
- CORTEX_CHUNK: Chunked, searchable content.

In [ ]:
-- inspect the results and count the tokens for each document
SELECT *, SNOWFLAKE.CORTEX.COUNT_TOKENS('mistral-7b', RAW_TEXT) as token_count
FROM {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARSED_TEXT;

In [ ]:
-- Chunk the text based on paragraph seperators and write into PARAGRAPH_CHUNKS;

CREATE OR REPLACE TABLE {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARAGRAPH_CHUNKS AS
WITH text_chunks AS (
    SELECT
        relative_path,
        SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
            raw_text:content::STRING,  -- Extract the 'content' field from the JSON
            'markdown', -- Adjust to 'markdown' if needed
            2000,       -- Adjust chunk size
            100,        -- Adjust overlap size
            ['\n\n']    -- Adjust separators
        ) AS chunks
    FROM {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARSED_TEXT
)
SELECT
    relative_path,
    c.value AS chunk  -- Extract each chunk of the parsed text
FROM text_chunks,
LATERAL FLATTEN(INPUT => chunks) c;

In [ ]:
SELECT * 
FROM {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARAGRAPH_CHUNKS
LIMIT 5;

## Step 2: Summarize at the document level

In [ ]:
-- Add the DOC_SUMMARY column if it doesn't exist
ALTER TABLE {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARAGRAPH_CHUNKS
    ADD COLUMN IF NOT EXISTS DOC_SUMMARY VARCHAR(5000);

-- Grouped summarization update
UPDATE {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARAGRAPH_CHUNKS AS tgt
SET DOC_SUMMARY = src.DOC_SUMMARY
FROM (
    SELECT 
        RELATIVE_PATH,
        AI_SUMMARIZE_AGG(CHUNK) AS DOC_SUMMARY
    FROM {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARAGRAPH_CHUNKS
    GROUP BY RELATIVE_PATH
) AS src
WHERE tgt.RELATIVE_PATH = src.RELATIVE_PATH
  AND tgt.DOC_SUMMARY IS NULL;


In [ ]:
ALTER TABLE {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARAGRAPH_CHUNKS ADD COLUMN IF NOT EXISTS CHUNK_WITH_SUMMARY VARCHAR;


UPDATE {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARAGRAPH_CHUNKS
  SET CHUNK_WITH_SUMMARY = DOC_SUMMARY || '\n\n' || CHUNK

## Step 3: Create Search Services
Next, create Cortex Search services with the two sets of chunks we want to compare:
- chunks with metadata
- contextualized chunks with metadata

Objectives: 
- Create two search services to compare performance

Key Outputs:
- SEARCH_SERVICE

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.SEC_CHUNK_RETRIEVAL
  ON SEARCH_COL
  WAREHOUSE = COMPUTE
  TARGET_LAG = '1 hour'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
  SELECT
    RELATIVE_PATH,
    CHUNK::STRING AS SEARCH_COL,
  FROM {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARAGRAPH_CHUNKS
);

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.SEC_CONTEXTUAL_CHUNK_RETRIEVAL
  ON SEARCH_COL
  WAREHOUSE = COMPUTE
  TARGET_LAG = '1 hour'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
  SELECT
    RELATIVE_PATH,
    CHUNK,
    CHUNK_WITH_SUMMARY::STRING AS SEARCH_COL
  FROM {{DB_NAME}}.{{TEXT_SCHEMA_NAME}}.PARAGRAPH_CHUNKS
);

In [ ]:
# Query your Snowflake Cortex Search Service using the Snowpark Python API to retrieve and process search results.
from snowflake.snowpark import Session
from snowflake.core import Root
root = Root(session)

sec_search_service = (root
  .databases[DB_NAME]
  .schemas[TEXT_SCHEMA_NAME]
  .cortex_search_services['SEC_CHUNK_RETRIEVAL']
)

resp = sec_search_service.search(
  query="""What was under armour's net sales?""",
  columns=['SEARCH_COL'],
  # filter={"@eq": {"MEETING_YEAR": "2020"} },
  limit=3
)
results = resp.results
results

In [ ]:
# Query your Snowflake Cortex Search Service using the Snowpark Python API to retrieve and process search results.
import streamlit as st
from snowflake.snowpark import Session
from snowflake.core import Root
root = Root(session)

sec_contextual_search_service = (root
  .databases[DB_NAME]
  .schemas[TEXT_SCHEMA_NAME]
  .cortex_search_services['SEC_CONTEXTUAL_CHUNK_RETRIEVAL']
)

resp = sec_contextual_search_service.search(
  query="""What was zscalar's net sales?""",
  columns=['SEARCH_COL'],
  # filter={"@eq": {"MEETING_YEAR": "2021"} },
  limit=3
)
results = resp.results
results

## Step 4: Create RAG
Next, create two versions of a RAG with each of the created search services so we can compare end to end performance.

Key Outputs:
- rag
- contextual_rag

In [ ]:
from snowflake.cortex import complete
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

class RAG:

    def __init__(self, search_service):
        self.search_service = search_service

    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
            }
    )
    def retrieve_context(self, query: str) -> list:
        """
        Retrieve relevant text from vector store.
        """
        
        response = self.search_service.search(
          query=query,
          columns=['SEARCH_COL'],
          limit=4
        )

        if response.results:
            return [curr["SEARCH_COL"] for curr in response.results]
        else:
            return []


    @instrument(
        span_type=SpanAttributes.SpanType.GENERATION)
    def generate_completion(self, query: str, context_list: list) -> str:
        """
        Generate answer from context.
        """
        prompt = f"""
          You are an expert assistant extracting information from context provided.
          Answer the question as concisely as possible without any preface.
          Context: {context_list}
          Question:
          {query}
          Answer:
        """
        response = complete("claude-4-sonnet", prompt)
        return response

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT, 
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "query",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        })
    def query(self, query: str) -> str:
        context_str = self.retrieve_context(query)
        return self.generate_completion(query, context_str)


rag = RAG(search_service = sec_search_service)
contextual_rag = RAG(search_service = sec_contextual_search_service)

## Step 5: Set up AI Observability to compare versions

[AI Observability in Snowflake](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability) enables you to easily evaluate and trace Generative AI applications, like the one you are building in this quickstart.

With AI Observability, you can measure the performance of your AI applications by running systematic evaluations, and iterate on your application configurations to optimize performance. Importantly, you can also directly compare different versions of your application to measure improvement. This is the workflow we'll follow here.

In this step, you will connect to Snowflake using the `Snowflake Connector` and then register two versions of your RAG, `base` and `conextual chunks`.

In [ ]:
from trulens.apps.app import TruApp
from trulens.connectors.snowflake import SnowflakeConnector

tru_snowflake_connector = SnowflakeConnector(snowpark_session=session)

In [ ]:
app_name = "sec_10k_chat_app"

base_tru_recorder = TruApp(
        rag,
        app_name=app_name,
        app_version="base",
        connector=tru_snowflake_connector,
        main_method_name="query"
    )

contextual_tru_recorder = TruApp(
        contextual_rag,
        app_name=app_name,
        app_version="contextual chunks",
        connector=tru_snowflake_connector,
        main_method_name="query"
    )

## Step 6: Create a dataset to test our app

Next, you need a dataset to test your RAG against. AI Observability can be run against both python dataframes and Snowflake tables. Here, we'll create a small test set as a table in Snowflake including questios and ground truth answers.

In [ ]:
CREATE OR REPLACE TABLE SEC_FILINGS_QA (
  COMPANY_NAME        STRING,
  QUESTION            STRING,
  GROUND_TRUTH_ANSWER STRING
);

INSERT INTO SEC_FILINGS_QA
  (COMPANY_NAME, QUESTION, GROUND_TRUTH_ANSWER)
VALUES
  (
    'Autodesk, Inc.',
    'In Autodesk’s Form 10-K covering fiscal year 2024, what amount of Remaining Performance Obligations (RPO) was reported as of the close of the fiscal year on January 31, 2024?',
    '$6.11 billion'
  ),
  (
    'IQVIA Holdings Inc.',
    'According to IQVIA’s most recent annual 10-K, what was the approximate total number of employees worldwide at the end of the reporting period?',
    'IQVIA has approximately 87,000 employees.'
  ),
  (
    'Alcoa Corporation',
    'Within Alcoa’s 2022 10-K, how many thousand metric tons of alumina shipments to third parties were disclosed for the six months ending June 30, 2022?',
    '4,715 kmt'
  ),
  (
    'Zscaler, Inc.',
    'As reported in Zscaler’s 2024 10-K, what was the total deferred revenue recorded at the fiscal year-end date of July 31, 2024?',
    '$1,895.0 million'
  ),
  (
    '3M Company',
    'In 3M’s discussion of environmental litigation matters in its 2018 Form 10-K, what was the pre-tax amount recorded in connection with the Minnesota settlement over PFAS?',
    '$897 million'
  ),
  (
    'Under Armour, Inc.',
    'Looking at Under Armour’s 2020 10-K, what net revenue and cost of goods sold figures were reported for the three-month period ending June 30, 2020?',
    'Net revenue: $707,640 thousand - Cost of goods sold: $358,471 thousand'
  ),
  (
    'Packaging Corporation of America',
    'According to PCA’s 2019 annual 10-K disclosures, what was the final purchase price paid to acquire the assets of Englander?',
    '$57.7 milion'
  ),
  (
    'Spectrum Brands Holdings',
    'In Spectrum’s 2022 10-K, which specific appliances and cookware business acquisition was highlighted in the notes to the financial statements?',
    'Tristar Business'
  ),
  (
    'Spectrum Brands Holdings',
    'Per Spectrum’s most recent 10-K description of its operating structure, what are the principal business segments reported?',
    'GPC (Global Pet Care), H&G (Home & Garden), and HPC (Home & Personal Care)'
  ),
  (
    'Southwestern Energy Company',
    'In the 2018 10-K, Southwestern Energy reported the divestiture of its Arkansas subsidiaries — which buyer purchased them and at what agreed price?',
    'Flywheel Energy Operating LLC, for a price of $1,650 million.'
  );


## Step 7: Create and Compare Runs

Now that we've registered the two versions of our RAG and have a dataset ready to test, we can configure our evaluation runs.

These runs will form the basis of our comparison.

In [ ]:
from trulens.core.run import Run
from trulens.core.run import RunConfig
from datetime import datetime

TEST_RUN_NAME = f"base_run_{datetime.now().strftime('%Y%m%d%H%M%S')}"

base_run_config = RunConfig(
    run_name=TEST_RUN_NAME,
    description="Questions about SEC 10KS",
    dataset_name="SEC_FILINGS_QA",
    source_type="TABLE",
    label="CHUNKS",
    dataset_spec={
        "RECORD_ROOT.INPUT": "QUESTION",
        "RECORD_ROOT.GROUND_TRUTH_OUTPUT":"GROUND_TRUTH_ANSWER",
    },
)

base_run = base_tru_recorder.add_run(run_config=base_run_config)

In [ ]:
CONTEXTUAL_TEST_RUN_NAME = f"contextual_run_{datetime.now().strftime('%Y%m%d%H%M%S')}"

contextual_run_config = RunConfig(
    run_name=CONTEXTUAL_TEST_RUN_NAME,
    dataset_name="SEC_FILINGS_QA",
    description="Questions about SEC 10KS",
    source_type="TABLE",
    label="CONTEXTUAL_CHUNKS",
    dataset_spec={
        "RECORD_ROOT.INPUT": "QUESTION",
        "RECORD_ROOT.GROUND_TRUTH_OUTPUT":"GROUND_TRUTH_ANSWER",
    }
)

contextual_run = contextual_tru_recorder.add_run(run_config=contextual_run_config)

### Start the runs

Starting the runs invokes the RAG in batch for the queries in the dataset.

In [ ]:
base_run.start()
print("Finished base run")

In [ ]:
contextual_run.start()
print("Finished contextual run")

In [ ]:
run_list = [base_run, contextual_run]

### Compute metrics

Once both versions of the RAG have run for all the questions in the test set, we can compute metrics. Here we will use both reference-free metrics (answer relevance, context relevance, and groundedness), along with correctness which leverages the ground truth we have available in the dataset.

In [ ]:
import time
for i in run_list:
    while i.get_status() == "INVOCATION_IN_PROGRESS":
        time.sleep(3)
    if i.get_status() == "INVOCATION_COMPLETED":
        i.compute_metrics(["correctness",
                           "answer_relevance",
                           "context_relevance",
                           "groundedness",])
        print(f"Kicked off Metrics Computation for Run {i.run_name}")
    if i.get_status() in ["FAILED", "UNKNOWN"]:
        print("Not able to compute metrics! Run status:", i.get_status())

### Compare results.

Finally, let's view the results of our runs by opening the leaderboard:

[AI Observability Leaderboard](https://app.snowflake.com/_deeplink/#/ai-evaluations/databases/CHUNKING_EVALUATION/schemas/PUBLIC/applications/SEC_10K_CHAT_APP?utm_source=quickstart&utm_medium=quickstart&utm_campaign=-us-en-all&utm_content=app-compare-and-evaluate-chunking-strategies)

We can also compare the runs head to head by clicking the checkbox for each version and then clicking `Compare` on the right side of the screen.